In [2]:
!pip install requests pandas numpy matplotlib scikit-learn streamlit

You should consider upgrading via the '/Users/yashvibhansali/Documents/Housing Affordability Research/.venv/bin/python3 -m pip install --upgrade pip' command.


In [3]:
import requests
import pandas as pd
          
dataset_id = "d_14f63e595975691e7c24a27ae4c07c79"
url = f"https://data.gov.sg/api/action/datastore_search?resource_id={dataset_id}&limit=10000"

response = requests.get(url)

data = response.json()
df = pd.DataFrame(data["result"]["records"])
df.head()


/Users/yashvibhansali/Documents/Housing Affordability Research/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,_id,quarter,index
0,1,1990-Q1,24.3
1,2,1990-Q2,24.4
2,3,1990-Q3,25
3,4,1990-Q4,24.7
4,5,1991-Q1,24.9


In [5]:
import os

# make sure the folder exists
os.makedirs("Data/Raw", exist_ok=True)

df.to_csv("Data/Raw/singapore_hdb_resale_price_index.csv", index=False)
print("Saved:", df.shape)

Saved: (146, 3)


In [24]:
import requests
import pandas as pd

dataset_id = "d_8b84c4ee58e3cfc0ece0d773c8ca6abc"
base_url = "https://data.gov.sg/api/action/datastore_search"

all_records = []
offset = 0
batch_size = 10000

while True:
    url = f"{base_url}?resource_id={dataset_id}&limit={batch_size}&offset={offset}"
    response = requests.get(url)
    data = response.json()
    
    records = data["result"]["records"]
    if not records:
        break
    
    all_records.extend(records)
    offset += batch_size
    print(f"Pulled {len(all_records)} so far...")

df = pd.DataFrame(all_records)
print(df.shape)
df.head()

Pulled 10000 so far...
Pulled 20000 so far...
Pulled 30000 so far...
Pulled 40000 so far...
Pulled 50000 so far...
Pulled 60000 so far...
Pulled 70000 so far...
Pulled 80000 so far...
Pulled 90000 so far...
Pulled 100000 so far...
Pulled 110000 so far...
Pulled 120000 so far...
Pulled 130000 so far...
Pulled 140000 so far...
Pulled 150000 so far...
Pulled 160000 so far...
Pulled 170000 so far...
Pulled 180000 so far...
Pulled 190000 so far...
Pulled 200000 so far...
Pulled 210000 so far...
Pulled 220000 so far...
Pulled 230000 so far...
Pulled 238932 so far...
(238932, 12)


,_id,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,1,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,Improved,1979,61 years 04 months,232000
1,2,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,New Generation,1978,60 years 07 months,250000
2,3,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,New Generation,1980,62 years 05 months,262000
3,4,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,New Generation,1980,62 years 01 month,265000
4,5,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,New Generation,1980,62 years 05 months,265000


In [25]:
df['quarter'] = pd.to_datetime(df['month']).dt.to_period('Q')
volume_by_quarter = df.groupby('quarter').size().reset_index(name='resale_volume')
volume_by_quarter.head()
print(volume_by_quarter.shape)

(39, 2)


In [26]:
volume_by_quarter.to_csv('Data/Raw/singapore_resale_volume.csv', index=False)

In [19]:
print(df.columns.tolist())
df['quarter'] = pd.to_datetime(df['month']).dt.to_period('Q')
volume_by_quarter = df.groupby('quarter').size().reset_index(name='resale_volume')
volume_by_quarter.head()
volume_by_quarter.shape

['_id', 'month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'quarter']


(3, 2)

In [17]:
volume_by_quarter.to_csv('Data/Raw/singapore_resale_volume.csv', index=False)

In [37]:
import pandas as pd

def load_singstat_csv(path):
    with open(path, 'r') as f:
        lines = f.readlines()
    
    header_row = None
    for i, line in enumerate(lines):
        first_cell = line.split(',')[0].strip().strip('"')
        if first_cell in ('Year', 'Quarter', 'Data Series'):
            header_row = i
            break
    
    if header_row is None:
        raise ValueError(f"Could not find header row in {path} — check file manually")
    
    print(f"{path}: header found at row {header_row}")
    df = pd.read_csv(path, skiprows=header_row)
    return df

income = load_singstat_csv('Data/Raw/singapore_avg&median_income.csv')
cpi = load_singstat_csv('Data/Raw/singapore_cpi.csv')
gdp = load_singstat_csv('Data/Raw/singapore_gdp_year_on_year.csv')

price_index = pd.read_csv('Data/Raw/singapore_hdb_resale_price_index.csv')
resale_volume = pd.read_csv('Data/Raw/singapore_resale_volume.csv')

for name, df in [('income', income), ('cpi', cpi), ('gdp', gdp),
                  ('price_index', price_index), ('resale_volume', resale_volume)]:
    print(f"\n{name}: {df.shape}")
    print(df.columns.tolist()[:10])
    print(df.head(3))

Data/Raw/singapore_avg&median_income.csv: header found at row 9
Data/Raw/singapore_cpi.csv: header found at row 10
Data/Raw/singapore_gdp_year_on_year.csv: header found at row 10

income: (35, 4)
['Year', 'Average', 'Median1/', 'Unnamed: 3']
   Year  Average  Median1/  Unnamed: 3
0  2015  12135.0    8839.0         NaN
1  2016  11886.0    8934.0         NaN
2  2017  12210.0    9027.0         NaN

cpi: (245, 263)
['Data Series', '2026 2Q', '2026 1Q', '2025 4Q', '2025 3Q', '2025 2Q', '2025 1Q', '2024 4Q', '2024 3Q', '2024 2Q']
                                    Data Series  2026 2Q  2026 1Q  2025 4Q  \
0                                     All Items  102.575  101.885  101.588   
1                                          Food  102.833  102.365  101.695   
2    Food Excl Food & Beverage Serving Services  103.442  102.979  101.573   

   2025 3Q  2025 2Q  2025 1Q  2024 4Q  2024 3Q  2024 2Q  ...  1963 2Q 1963 1Q  \
0  100.910  100.720  100.395  100.389  100.292   99.913  ...   21.241  21.58

In [38]:
# --- Clean income ---
income = income[['Year', 'Average', 'Median1/']].copy()
income = income.rename(columns={'Median1/': 'Median'})
print(income.head())

# --- Reshape CPI (wide -> long) ---
# Pick the "All Items" row — the headline CPI figure
cpi_all_items = cpi[cpi['Data Series'] == 'All Items']

# Drop the label column, transpose so periods become rows
cpi_long = cpi_all_items.drop(columns=['Data Series']).T.reset_index()
cpi_long.columns = ['quarter', 'cpi']
cpi_long = cpi_long.sort_values('quarter').reset_index(drop=True)
print(cpi_long.head())
print(cpi_long.tail())

# --- Reshape GDP (same wide -> long pattern) ---
print(gdp['Data Series'].unique()[:15])  # check exact label to filter on

   Year  Average  Median
0  2015  12135.0  8839.0
1  2016  11886.0  8934.0
2  2017  12210.0  9027.0
3  2018  12362.0  9207.0
4  2019  12451.0  9232.0
   quarter     cpi
0  1961 1Q  21.092
1  1961 2Q  20.742
2  1961 3Q   20.95
3  1961 4Q  21.029
4  1962 1Q  21.135
     quarter      cpi
257  2025 2Q   100.72
258  2025 3Q   100.91
259  2025 4Q  101.588
260  2026 1Q  101.885
261  2026 2Q  102.575
['GDP At Current Market Prices' '  Goods Producing Industries'
 '    Manufacturing' '    Construction' '    Utilities'
 '    Other Goods Industries' '  Services Producing Industries'
 '    Wholesale & Retail Trade' '      Wholesale Trade'
 '      Retail Trade' '    Transportation & Storage'
 '    Accommodation & Food Services' '      Accommodation'
 '      Food & Beverage Services' '    Information & Communications']


In [39]:
# --- Reshape GDP (same wide -> long pattern as CPI) ---
gdp_total = gdp[gdp['Data Series'] == 'GDP At Current Market Prices']
gdp_long = gdp_total.drop(columns=['Data Series']).T.reset_index()
gdp_long.columns = ['quarter', 'gdp']
gdp_long = gdp_long.sort_values('quarter').reset_index(drop=True)
print(gdp_long.head())
print(gdp_long.tail())

    quarter   gdp
0  1976 1Q   10.5
1  1976 2Q    9.5
2  1976 3Q    9.3
3  1976 4Q    8.1
4  1977 1Q    9.3
      quarter   gdp
197  2025 2Q    3.4
198  2025 3Q    0.7
199  2025 4Q    3.2
200  2026 1Q    7.8
201  2026 2Q   12.4


In [40]:
# --- Standardize CPI and GDP quarter format: "2017 1Q" -> "2017Q1" ---
def fix_quarter_format(q):
    year, qtr = q.split()          # "2017 1Q" -> "2017", "1Q"
    qtr_num = qtr.replace('Q', '') # "1Q" -> "1"
    return f"{year}Q{qtr_num}"

cpi_long['quarter'] = cpi_long['quarter'].apply(fix_quarter_format)
gdp_long['quarter'] = gdp_long['quarter'].apply(fix_quarter_format)

# --- Standardize price_index quarter format too — check what it looks like first ---
print(price_index.head())

   _id  quarter  index
0    1  1990-Q1   24.3
1    2  1990-Q2   24.4
2    3  1990-Q3   25.0
3    4  1990-Q4   24.7
4    5  1991-Q1   24.9


In [42]:
print(income.tail(10))
print(income['Year'].isna().sum())

                                                 Year  Average  Median
25                       nes  not elsewhere specified      NaN     NaN
26         -     nil or negligible or not significant      NaN     NaN
27                                              Notes      NaN     NaN
28  Numbers may not add up to the totals due to ro...      NaN     NaN
29  Data are the latest available at the time of a...      NaN     NaN
30  Values are shown in Singapore dollars (unless ...      NaN     NaN
31                                                NaN      NaN     NaN
32               Generated by: SingStat Table Builder      NaN     NaN
33                         Date generated: 24/08/2026      NaN     NaN
34                      Contact: info@singstat.gov.sg      NaN     NaN
5


In [44]:
# Keep only rows where Year is actually a valid number
income = income[pd.to_numeric(income['Year'], errors='coerce').notna()].copy()
income['Year'] = income['Year'].astype(int)
print(income.shape)
print(income.tail())

(11, 3)
    Year  Average   Median
6   2021  12948.0   9731.0
7   2022  13721.0  10305.0
8   2023  14645.0  11130.0
9   2024  15303.0  11558.0
10  2025  16159.0  12446.0


In [45]:
# --- Fix price_index quarter format: "1990-Q1" -> "1990Q1" ---
price_index['quarter'] = price_index['quarter'].str.replace('-', '', regex=False)
print(price_index.head())

# --- Expand income from annual to quarterly ---
income_quarterly = []
for _, row in income.iterrows():
    for q in [1, 2, 3, 4]:
        income_quarterly.append({
            'quarter': f"{int(row['Year'])}Q{q}",
            'income_avg': row['Average'],
            'income_median': row['Median']
        })
income_long = pd.DataFrame(income_quarterly)
print(income_long.head(8))

# --- Now merge everything on 'quarter' ---
merged = price_index[['quarter', 'index']].rename(columns={'index': 'resale_price_index'})
merged = merged.merge(cpi_long, on='quarter', how='inner')
merged = merged.merge(gdp_long, on='quarter', how='inner')
merged = merged.merge(income_long, on='quarter', how='inner')
merged = merged.merge(resale_volume, on='quarter', how='inner')

print(merged.shape)
merged.head()
merged.tail()

   _id quarter  index
0    1  1990Q1   24.3
1    2  1990Q2   24.4
2    3  1990Q3   25.0
3    4  1990Q4   24.7
4    5  1991Q1   24.9
  quarter  income_avg  income_median
0  2015Q1     12135.0         8839.0
1  2015Q2     12135.0         8839.0
2  2015Q3     12135.0         8839.0
3  2015Q4     12135.0         8839.0
4  2016Q1     11886.0         8934.0
5  2016Q2     11886.0         8934.0
6  2016Q3     11886.0         8934.0
7  2016Q4     11886.0         8934.0
(36, 7)


,quarter,resale_price_index,cpi,gdp,income_avg,income_median,resale_volume
31,2024Q4,197.9,100.389,12.8,15303.0,11558.0,6180
32,2025Q1,201.0,100.395,5.5,16159.0,12446.0,6303
33,2025Q2,202.9,100.72,3.4,16159.0,12446.0,6823
34,2025Q3,203.7,100.91,0.7,16159.0,12446.0,6939
35,2025Q4,203.6,101.588,3.2,16159.0,12446.0,5020


In [46]:
print(merged['quarter'].min(), '-', merged['quarter'].max())
merged.head()
merged.tail()
merged.isnull().sum()

2017Q1 - 2025Q4


quarter               0
resale_price_index    0
cpi                   0
gdp                   0
income_avg            0
income_median         0
resale_volume         0
dtype: int64

In [47]:
merged.to_csv('Data/Raw/singapore_merged_draft.csv', index=False)

In [4]:
merged = pd.read_csv('Data/Raw/singapore_merged_draft.csv')

print(merged.shape)
print(merged.dtypes)
merged.describe()

(36, 7)
quarter                object
resale_price_index    float64
cpi                   float64
gdp                   float64
income_avg            float64
income_median         float64
resale_volume           int64
dtype: object


,resale_price_index,cpi,gdp,income_avg,income_median,resale_volume
count,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,157.208333,91.310167,7.097222,13577.222222,10192.777778,6168.527778
std,26.810663,6.426208,10.370769,1402.461165,1195.003864,1064.956055
min,130.800000,85.028000,-14.200000,12210.000000,9027.000000,3225.000000
25%,131.850000,85.682250,0.500000,12396.000000,9207.000000,5542.000000
50%,148.500000,87.591000,5.900000,12948.000000,9731.000000,6321.500000
75%,178.975000,98.364250,9.750000,14645.000000,11130.000000,6852.000000
max,203.700000,101.588000,33.700000,16159.000000,12446.000000,7914.000000


In [9]:
expected = pd.period_range('2017Q1', '2025Q4', freq='Q').astype(str).str.replace('Q', 'Q')
actual = set(merged['quarter'])
missing_quarters = [q for q in expected.tolist() if q not in actual]
print(missing_quarters)

import os
os.makedirs('Data/Clean', exist_ok=True)
merged.to_csv('Data/Clean/singapore_merged.csv', index=False)

[]
